# Building Energy Data for the Energy Consumption Forecasting Case Study

To work with this notebook, you need the `building_energy_data.csv` file. Here are two options to get this data:

## Option 1: Download Sample Dataset

You can download sample building energy data from a public repository:

```python
import pandas as pd
import os

# URL to download the dataset
url = "https://raw.githubusercontent.com/buds-lab/building-data-genome-project-2/master/data/meters/cleaned/electricity_meters.csv"

# Download and process the data
def download_building_energy_data():
    print("Downloading building energy data...")
    
    # Download the raw data
    raw_data = pd.read_csv(url)
    
    # Select one building (most suitable for the tutorial)
    building_id = 'OG_2'
    energy_data = raw_data[['timestamp', building_id]].rename(columns={building_id: 'energy_consumption'})
    
    # Convert timestamp to datetime
    energy_data['timestamp'] = pd.to_datetime(energy_data['timestamp'])
    
    # Generate weather data (since original dataset doesn't include it)
    # Create synthetic temperature data with seasonal patterns
    dates = energy_data['timestamp']
    energy_data['outdoor_temperature'] = 15 + 10 * np.sin((dates.dt.dayofyear / 365) * 2 * np.pi) + 5 * np.random.randn(len(dates))
    energy_data['humidity'] = 60 + 10 * np.sin((dates.dt.dayofyear / 182) * 2 * np.pi) + 15 * np.random.randn(len(dates))
    energy_data['occupancy'] = np.where((dates.dt.hour >= 9) & (dates.dt.hour <= 17) & (dates.dt.dayofweek < 5), 
                                        0.7 + 0.3 * np.random.rand(len(dates)), 
                                        0.1 + 0.2 * np.random.rand(len(dates)))
    
    # Save the processed dataset
    energy_data.to_csv('building_energy_data.csv', index=False)
    print(f"Data saved to 'building_energy_data.csv' with {len(energy_data)} rows")
    return energy_data

# Download the data if it doesn't exist
if not os.path.exists('building_energy_data.csv'):
    energy_data = download_building_energy_data()
else:
    print("File 'building_energy_data.csv' already exists. Loading existing file.")
    energy_data = pd.read_csv('building_energy_data.csv', parse_dates=['timestamp'])
```

## Option 2: Generate Synthetic Data

Alternatively, you can generate realistic synthetic building energy data:

```python
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

def generate_synthetic_energy_data():
    print("Generating synthetic building energy data...")
    
    # Generate time series for 2 years of hourly data
    start_date = datetime(2021, 1, 1)
    end_date = datetime(2022, 12, 31, 23)
    dates = pd.date_range(start=start_date, end=end_date, freq='H')
    
    # Create DataFrame with timestamp
    df = pd.DataFrame({'timestamp': dates})
    
    # Basic patterns
    hour_of_day = df['timestamp'].dt.hour
    day_of_week = df['timestamp'].dt.dayofweek
    month_of_year = df['timestamp'].dt.month
    
    # Generate outdoor temperature with seasonal patterns
    seasonal_temp = 15 + 10 * np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi)
    daily_temp_variation = 5 * np.sin((hour_of_day / 24) * 2 * np.pi - np.pi/2)
    random_temp_variation = 3 * np.random.randn(len(df))
    df['outdoor_temperature'] = seasonal_temp + daily_temp_variation + random_temp_variation
    
    # Generate humidity
    df['humidity'] = 60 + 20 * np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi - np.pi/4) + 10 * np.random.randn(len(df))
    df['humidity'] = np.clip(df['humidity'], 20, 95)
    
    # Building occupancy (workdays 8am-6pm)
    is_workday = (day_of_week < 5)
    is_work_hours = (hour_of_day >= 8) & (hour_of_day <= 18)
    occupancy_base = np.where(is_workday & is_work_hours, 0.7, 0.1)
    df['occupancy'] = occupancy_base + 0.2 * np.random.random(len(df))
    
    # Energy consumption
    # Base load
    base_load = 20 + 5 * np.random.random(len(df))
    
    # HVAC load (more when temp is far from 21°C comfort temp)
    hvac_load = 5 + 0.5 * np.abs(df['outdoor_temperature'] - 21)**1.5 
    
    # Lighting & equipment (varies by hour and occupancy)
    lighting_equip = 5 + 20 * df['occupancy']
    
    # Seasonal adjustment (more energy in winter)
    winter_effect = 15 * (1 - np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi))
    
    # Hour of day effect
    hour_effect = 10 * np.sin((hour_of_day / 24) * 2 * np.pi)
    
    # Weekend effect (less energy on weekends)
    weekend_effect = np.where(day_of_week >= 5, 0.7, 1.0)
    
    # Combine all effects
    df['energy_consumption'] = (base_load + hvac_load + lighting_equip + winter_effect + hour_effect) * weekend_effect
    
    # Add some random noise
    df['energy_consumption'] += 5 * np.random.randn(len(df))
    df['energy_consumption'] = np.maximum(df['energy_consumption'], 10)  # Ensure minimum value
    
    # Add some features that might be useful
    df['wind_speed'] = 5 + 10 * np.random.random(len(df))
    df['solar_radiation'] = np.maximum(0, 
                                      np.sin((hour_of_day - 6) * np.pi / 12) * 
                                      (1 - 0.3 * np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi - np.pi))) * 1000
    df['solar_radiation'] = np.where((hour_of_day >= 19) | (hour_of_day <= 6), 0, df['solar_radiation'])
    df['solar_radiation'] += np.random.random(len(df)) * 50
    
    # Save to CSV
    df.to_csv('building_energy_data.csv', index=False)
    print(f"Synthetic data saved to 'building_energy_data.csv' with {len(df)} rows")
    return df

# Generate synthetic data if file doesn't exist
if not os.path.exists('building_energy_data.csv'):
    energy_data = generate_synthetic_energy_data()
else:
    print("File 'building_energy_data.csv' already exists. Loading existing file.")
    energy_data = pd.read_csv('building_energy_data.csv', parse_dates=['timestamp'])
```

## Using the Data in the Notebook

After running either of the above code blocks, you can load and use the data in the notebook with:

```python
# If you've just generated the data, you can use it directly
# If you're loading from the CSV:
energy_data = pd.read_csv('building_energy_data.csv', parse_dates=['timestamp'])

# Set timestamp as index
energy_data.set_index('timestamp', inplace=True)

# Display basic information
print(f"Dataset shape: {energy_data.shape}")
print(f"Date range: {energy_data.index.min()} to {energy_data.index.max()}")
print(f"Missing values: {energy_data.isnull().sum().sum()}")

# Preview the data
print("\nData preview:")
print(energy_data.head())
```

Choose Option 2 (synthetic data) if you prefer to work with locally generated data or if you have internet connectivity issues. The synthetic data is designed to reflect realistic patterns in building energy consumption, including daily and seasonal variations, weekday/weekend differences, and temperature dependencies.

# Building Energy Data for Energy Consumption Forecasting

I checked the URL and it returns a 404 error. Here are corrected options for getting the building energy data:

## Option 1: Use the Correct Building Data Genome Project URL

The correct URL for the Building Data Genome Project 2 dataset is:

```python
import pandas as pd
import numpy as np
import os

# Corrected URL for the Building Data Genome Project 2
url = "https://github.com/buds-lab/building-data-genome-project-2/raw/master/data/meter_data/cleaned/electricity_cleaned.csv"

# Download and process the data
def download_building_energy_data():
    print("Downloading building energy data...")
    
    # Download the raw data
    raw_data = pd.read_csv(url)
    
    # Select one building (most suitable for the tutorial)
    building_id = 'LBNL_1'
    energy_data = raw_data[['timestamp', building_id]].rename(columns={building_id: 'energy_consumption'})
    
    # Convert timestamp to datetime
    energy_data['timestamp'] = pd.to_datetime(energy_data['timestamp'])
    
    # Generate weather data (since original dataset doesn't include it)
    dates = energy_data['timestamp']
    energy_data['outdoor_temperature'] = 15 + 10 * np.sin((dates.dt.dayofyear / 365) * 2 * np.pi) + 5 * np.random.randn(len(dates))
    energy_data['humidity'] = 60 + 10 * np.sin((dates.dt.dayofyear / 182) * 2 * np.pi) + 15 * np.random.randn(len(dates))
    energy_data['occupancy'] = np.where((dates.dt.hour >= 9) & (dates.dt.hour <= 17) & (dates.dt.dayofweek < 5), 
                                      0.7 + 0.3 * np.random.rand(len(dates)), 
                                      0.1 + 0.2 * np.random.rand(len(dates)))
    
    # Save the processed dataset
    energy_data.to_csv('building_energy_data.csv', index=False)
    print(f"Data saved to 'building_energy_data.csv' with {len(energy_data)} rows")
    return energy_data
```

## Option 2: Use the ASHRAE Great Energy Predictor Dataset

```python
import pandas as pd
import numpy as np
import os

# ASHRAE Great Energy Predictor III dataset
def download_ashrae_data():
    print("Downloading ASHRAE building energy data...")
    
    # URL for ASHRAE dataset (building 0)
    url = "https://raw.githubusercontent.com/buds-lab/ashrae-great-energy-predictor-3-solution-analysis/master/data/train/building_0.csv"
    
    try:
        # Download the dataset
        energy_data = pd.read_csv(url)
        
        # Rename columns for consistency with our code
        energy_data = energy_data.rename(columns={
            'meter_reading': 'energy_consumption',
            'air_temperature': 'outdoor_temperature'
        })
        
        # Convert timestamp
        energy_data['timestamp'] = pd.to_datetime(energy_data['timestamp'])
        
        # Add humidity if not present
        if 'humidity' not in energy_data.columns:
            dates = energy_data['timestamp']
            energy_data['humidity'] = 60 + 10 * np.sin((dates.dt.dayofyear / 182) * 2 * np.pi) + 15 * np.random.randn(len(dates))
        
        # Add occupancy approximation
        energy_data['occupancy'] = np.where(
            (energy_data['timestamp'].dt.hour >= 9) & 
            (energy_data['timestamp'].dt.hour <= 17) & 
            (energy_data['timestamp'].dt.dayofweek < 5),
            0.7 + 0.3 * np.random.rand(len(energy_data)), 
            0.1 + 0.2 * np.random.rand(len(energy_data))
        )
        
        # Save to CSV
        energy_data.to_csv('building_energy_data.csv', index=False)
        print(f"Data saved to 'building_energy_data.csv' with {len(energy_data)} rows")
        return energy_data
        
    except Exception as e:
        print(f"Error downloading ASHRAE data: {e}")
        print("Falling back to synthetic data generation...")
        return generate_synthetic_energy_data()
```

## Option 3: Use Synthetic Data (Most Reliable Option)

```python
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

def generate_synthetic_energy_data():
    print("Generating synthetic building energy data...")
    
    # Generate time series for 2 years of hourly data
    start_date = datetime(2021, 1, 1)
    end_date = datetime(2022, 12, 31, 23)
    dates = pd.date_range(start=start_date, end=end_date, freq='H')
    
    # Create DataFrame with timestamp
    df = pd.DataFrame({'timestamp': dates})
    
    # Basic patterns
    hour_of_day = df['timestamp'].dt.hour
    day_of_week = df['timestamp'].dt.dayofweek
    month_of_year = df['timestamp'].dt.month
    
    # Generate outdoor temperature with seasonal patterns
    seasonal_temp = 15 + 10 * np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi)
    daily_temp_variation = 5 * np.sin((hour_of_day / 24) * 2 * np.pi - np.pi/2)
    random_temp_variation = 3 * np.random.randn(len(df))
    df['outdoor_temperature'] = seasonal_temp + daily_temp_variation + random_temp_variation
    
    # Generate humidity
    df['humidity'] = 60 + 20 * np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi - np.pi/4) + 10 * np.random.randn(len(df))
    df['humidity'] = np.clip(df['humidity'], 20, 95)
    
    # Building occupancy (workdays 8am-6pm)
    is_workday = (day_of_week < 5)
    is_work_hours = (hour_of_day >= 8) & (hour_of_day <= 18)
    occupancy_base = np.where(is_workday & is_work_hours, 0.7, 0.1)
    df['occupancy'] = occupancy_base + 0.2 * np.random.random(len(df))
    
    # Energy consumption
    # Base load
    base_load = 20 + 5 * np.random.random(len(df))
    
    # HVAC load (more when temp is far from 21°C comfort temp)
    hvac_load = 5 + 0.5 * np.abs(df['outdoor_temperature'] - 21)**1.5 
    
    # Lighting & equipment (varies by hour and occupancy)
    lighting_equip = 5 + 20 * df['occupancy']
    
    # Seasonal adjustment (more energy in winter)
    winter_effect = 15 * (1 - np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi))
    
    # Hour of day effect
    hour_effect = 10 * np.sin((hour_of_day / 24) * 2 * np.pi)
    
    # Weekend effect (less energy on weekends)
    weekend_effect = np.where(day_of_week >= 5, 0.7, 1.0)
    
    # Combine all effects
    df['energy_consumption'] = (base_load + hvac_load + lighting_equip + winter_effect + hour_effect) * weekend_effect
    
    # Add some random noise
    df['energy_consumption'] += 5 * np.random.randn(len(df))
    df['energy_consumption'] = np.maximum(df['energy_consumption'], 10)  # Ensure minimum value
    
    # Save to CSV
    df.to_csv('building_energy_data.csv', index=False)
    print(f"Synthetic data saved to 'building_energy_data.csv' with {len(df)} rows")
    return df
```

## Complete Code: Try Multiple Sources and Fall Back to Synthetic

This comprehensive solution will try multiple data sources and fall back to synthetic data if needed:

```python
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import requests
from io import StringIO

def get_building_energy_data():
    """Get building energy data, trying multiple sources and falling back to synthetic data"""
    
    # Try multiple data sources in order
    data_sources = [
        {"name": "Building Data Genome 2", 
         "url": "https://github.com/buds-lab/building-data-genome-project-2/raw/master/data/meter_data/cleaned/electricity_cleaned.csv"},
        {"name": "ASHRAE Great Energy Predictor", 
         "url": "https://raw.githubusercontent.com/buds-lab/ashrae-great-energy-predictor-3-solution-analysis/master/data/train/building_0.csv"}
    ]
    
    # If file already exists, load it
    if os.path.exists('building_energy_data.csv'):
        print("File 'building_energy_data.csv' already exists. Loading existing file.")
        return pd.read_csv('building_energy_data.csv', parse_dates=['timestamp'])
    
    # Try each data source
    for source in data_sources:
        try:
            print(f"Attempting to download data from {source['name']}...")
            response = requests.get(source['url'], timeout=10)
            
            if response.status_code == 200:
                print(f"Successfully downloaded data from {source['name']}")
                
                # Process data based on source
                if "building-data-genome" in source['url']:
                    return process_bdg2_data(response.content)
                elif "ashrae" in source['url']:
                    return process_ashrae_data(response.content)
            else:
                print(f"Failed to download from {source['name']} (Status code: {response.status_code})")
                
        except Exception as e:
            print(f"Error accessing {source['name']}: {e}")
    
    # If all sources fail, generate synthetic data
    print("All data sources failed. Generating synthetic data instead.")
    return generate_synthetic_energy_data()

def process_bdg2_data(content):
    """Process Building Data Genome 2 dataset"""
    data = pd.read_csv(StringIO(content.decode('utf-8')))
    
    # Select one building
    building_id = list(data.columns)[1]  # First column after timestamp
    energy_data = data[['timestamp', building_id]].rename(columns={building_id: 'energy_consumption'})
    
    # Convert timestamp to datetime
    energy_data['timestamp'] = pd.to_datetime(energy_data['timestamp'])
    
    # Generate weather data (since original dataset doesn't include it)
    dates = energy_data['timestamp']
    energy_data['outdoor_temperature'] = 15 + 10 * np.sin((dates.dt.dayofyear / 365) * 2 * np.pi) + 5 * np.random.randn(len(dates))
    energy_data['humidity'] = 60 + 10 * np.sin((dates.dt.dayofyear / 182) * 2 * np.pi) + 15 * np.random.randn(len(dates))
    energy_data['occupancy'] = np.where((dates.dt.hour >= 9) & (dates.dt.hour <= 17) & (dates.dt.dayofweek < 5), 
                                      0.7 + 0.3 * np.random.rand(len(dates)), 
                                      0.1 + 0.2 * np.random.rand(len(dates)))
    
    # Save the processed dataset
    energy_data.to_csv('building_energy_data.csv', index=False)
    print(f"Data saved to 'building_energy_data.csv' with {len(energy_data)} rows")
    return energy_data

def process_ashrae_data(content):
    """Process ASHRAE data"""
    data = pd.read_csv(StringIO(content.decode('utf-8')))
    
    # Rename columns for consistency
    if 'meter_reading' in data.columns:
        data = data.rename(columns={'meter_reading': 'energy_consumption'})
    if 'air_temperature' in data.columns:
        data = data.rename(columns={'air_temperature': 'outdoor_temperature'})
    
    # Ensure all required columns exist
    if 'timestamp' not in data.columns:
        data['timestamp'] = pd.date_range(start='2020-01-01', periods=len(data), freq='H')
    else:
        data['timestamp'] = pd.to_datetime(data['timestamp'])
        
    if 'energy_consumption' not in data.columns:
        # Use any available energy column or create synthetic
        energy_cols = [col for col in data.columns if 'energy' in col.lower() or 'meter' in col.lower()]
        if energy_cols:
            data['energy_consumption'] = data[energy_cols[0]]
        else:
            # Generate synthetic energy consumption
            data['energy_consumption'] = 50 + 30 * np.sin((data['timestamp'].dt.hour / 24) * 2 * np.pi) + 20 * np.random.rand(len(data))
    
    if 'outdoor_temperature' not in data.columns:
        dates = data['timestamp']
        data['outdoor_temperature'] = 15 + 10 * np.sin((dates.dt.dayofyear / 365) * 2 * np.pi) + 5 * np.random.randn(len(dates))
    
    if 'humidity' not in data.columns:
        dates = data['timestamp']
        data['humidity'] = 60 + 10 * np.sin((dates.dt.dayofyear / 182) * 2 * np.pi) + 15 * np.random.randn(len(dates))
    
    if 'occupancy' not in data.columns:
        data['occupancy'] = np.where((data['timestamp'].dt.hour >= 9) & 
                                    (data['timestamp'].dt.hour <= 17) & 
                                    (data['timestamp'].dt.dayofweek < 5),
                                    0.7 + 0.3 * np.random.rand(len(data)), 
                                    0.1 + 0.2 * np.random.rand(len(data)))
    
    # Select only needed columns
    energy_data = data[['timestamp', 'energy_consumption', 'outdoor_temperature', 'humidity', 'occupancy']]
    
    # Save the processed dataset
    energy_data.to_csv('building_energy_data.csv', index=False)
    print(f"Data saved to 'building_energy_data.csv' with {len(energy_data)} rows")
    return energy_data

def generate_synthetic_energy_data():
    """Generate synthetic building energy data"""
    print("Generating synthetic building energy data...")
    
    # Generate time series for 2 years of hourly data
    start_date = datetime(2021, 1, 1)
    end_date = datetime(2022, 12, 31, 23)
    dates = pd.date_range(start=start_date, end=end_date, freq='H')
    
    # Create DataFrame with timestamp
    df = pd.DataFrame({'timestamp': dates})
    
    # Basic patterns
    hour_of_day = df['timestamp'].dt.hour
    day_of_week = df['timestamp'].dt.dayofweek
    month_of_year = df['timestamp'].dt.month
    
    # Generate outdoor temperature with seasonal patterns
    seasonal_temp = 15 + 10 * np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi)
    daily_temp_variation = 5 * np.sin((hour_of_day / 24) * 2 * np.pi - np.pi/2)
    random_temp_variation = 3 * np.random.randn(len(df))
    df['outdoor_temperature'] = seasonal_temp + daily_temp_variation + random_temp_variation
    
    # Generate humidity
    df['humidity'] = 60 + 20 * np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi - np.pi/4) + 10 * np.random.randn(len(df))
    df['humidity'] = np.clip(df['humidity'], 20, 95)
    
    # Building occupancy (workdays 8am-6pm)
    is_workday = (day_of_week < 5)
    is_work_hours = (hour_of_day >= 8) & (hour_of_day <= 18)
    occupancy_base = np.where(is_workday & is_work_hours, 0.7, 0.1)
    df['occupancy'] = occupancy_base + 0.2 * np.random.random(len(df))
    
    # Energy consumption
    # Base load
    base_load = 20 + 5 * np.random.random(len(df))
    
    # HVAC load (more when temp is far from 21°C comfort temp)
    hvac_load = 5 + 0.5 * np.abs(df['outdoor_temperature'] - 21)**1.5 
    
    # Lighting & equipment (varies by hour and occupancy)
    lighting_equip = 5 + 20 * df['occupancy']
    
    # Seasonal adjustment (more energy in winter)
    winter_effect = 15 * (1 - np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi))
    
    # Hour of day effect
    hour_effect = 10 * np.sin((hour_of_day / 24) * 2 * np.pi)
    
    # Weekend effect (less energy on weekends)
    weekend_effect = np.where(day_of_week >= 5, 0.7, 1.0)
    
    # Combine all effects
    df['energy_consumption'] = (base_load + hvac_load + lighting_equip + winter_effect + hour_effect) * weekend_effect
    
    # Add some random noise
    df['energy_consumption'] += 5 * np.random.randn(len(df))
    df['energy_consumption'] = np.maximum(df['energy_consumption'], 10)  # Ensure minimum value
    
    # Save to CSV
    df.to_csv('building_energy_data.csv', index=False)
    print(f"Synthetic data saved to 'building_energy_data.csv' with {len(df)} rows")
    return df

# Call the function to get the data
energy_data = get_building_energy_data()

# Display basic info
print(f"Dataset shape: {energy_data.shape}")
print(f"Date range: {energy_data['timestamp'].min()} to {energy_data['timestamp'].max()}")
print("\nData preview:")
print(energy_data.head())
```

# CREATE 'building_energy_data.csv'

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import requests
from io import StringIO

def get_building_energy_data():
    """Get building energy data, trying multiple sources and falling back to synthetic data"""
    
    # Try multiple data sources in order
    data_sources = [
        {"name": "Building Data Genome 2", 
         "url": "https://github.com/buds-lab/building-data-genome-project-2/raw/master/data/meter_data/cleaned/electricity_cleaned.csv"},
        {"name": "ASHRAE Great Energy Predictor", 
         "url": "https://raw.githubusercontent.com/buds-lab/ashrae-great-energy-predictor-3-solution-analysis/master/data/train/building_0.csv"}
    ]
    
    # If file already exists, load it
    if os.path.exists('building_energy_data.csv'):
        print("File 'building_energy_data.csv' already exists. Loading existing file.")
        return pd.read_csv('building_energy_data.csv', parse_dates=['timestamp'])
    
    # Try each data source
    for source in data_sources:
        try:
            print(f"Attempting to download data from {source['name']}...")
            response = requests.get(source['url'], timeout=10)
            
            if response.status_code == 200:
                print(f"Successfully downloaded data from {source['name']}")
                
                # Process data based on source
                if "building-data-genome" in source['url']:
                    return process_bdg2_data(response.content)
                elif "ashrae" in source['url']:
                    return process_ashrae_data(response.content)
            else:
                print(f"Failed to download from {source['name']} (Status code: {response.status_code})")
                
        except Exception as e:
            print(f"Error accessing {source['name']}: {e}")
    
    # If all sources fail, generate synthetic data
    print("All data sources failed. Generating synthetic data instead.")
    return generate_synthetic_energy_data()

def process_bdg2_data(content):
    """Process Building Data Genome 2 dataset"""
    data = pd.read_csv(StringIO(content.decode('utf-8')))
    
    # Select one building
    building_id = list(data.columns)[1]  # First column after timestamp
    energy_data = data[['timestamp', building_id]].rename(columns={building_id: 'energy_consumption'})
    
    # Convert timestamp to datetime
    energy_data['timestamp'] = pd.to_datetime(energy_data['timestamp'])
    
    # Generate weather data (since original dataset doesn't include it)
    dates = energy_data['timestamp']
    energy_data['outdoor_temperature'] = 15 + 10 * np.sin((dates.dt.dayofyear / 365) * 2 * np.pi) + 5 * np.random.randn(len(dates))
    energy_data['humidity'] = 60 + 10 * np.sin((dates.dt.dayofyear / 182) * 2 * np.pi) + 15 * np.random.randn(len(dates))
    energy_data['occupancy'] = np.where((dates.dt.hour >= 9) & (dates.dt.hour <= 17) & (dates.dt.dayofweek < 5), 
                                      0.7 + 0.3 * np.random.rand(len(dates)), 
                                      0.1 + 0.2 * np.random.rand(len(dates)))
    
    # Save the processed dataset
    energy_data.to_csv('building_energy_data.csv', index=False)
    print(f"Data saved to 'building_energy_data.csv' with {len(energy_data)} rows")
    return energy_data

def process_ashrae_data(content):
    """Process ASHRAE data"""
    data = pd.read_csv(StringIO(content.decode('utf-8')))
    
    # Rename columns for consistency
    if 'meter_reading' in data.columns:
        data = data.rename(columns={'meter_reading': 'energy_consumption'})
    if 'air_temperature' in data.columns:
        data = data.rename(columns={'air_temperature': 'outdoor_temperature'})
    
    # Ensure all required columns exist
    if 'timestamp' not in data.columns:
        data['timestamp'] = pd.date_range(start='2020-01-01', periods=len(data), freq='h')
    else:
        data['timestamp'] = pd.to_datetime(data['timestamp'])
        
    if 'energy_consumption' not in data.columns:
        # Use any available energy column or create synthetic
        energy_cols = [col for col in data.columns if 'energy' in col.lower() or 'meter' in col.lower()]
        if energy_cols:
            data['energy_consumption'] = data[energy_cols[0]]
        else:
            # Generate synthetic energy consumption
            data['energy_consumption'] = 50 + 30 * np.sin((data['timestamp'].dt.hour / 24) * 2 * np.pi) + 20 * np.random.rand(len(data))
    
    if 'outdoor_temperature' not in data.columns:
        dates = data['timestamp']
        data['outdoor_temperature'] = 15 + 10 * np.sin((dates.dt.dayofyear / 365) * 2 * np.pi) + 5 * np.random.randn(len(dates))
    
    if 'humidity' not in data.columns:
        dates = data['timestamp']
        data['humidity'] = 60 + 10 * np.sin((dates.dt.dayofyear / 182) * 2 * np.pi) + 15 * np.random.randn(len(dates))
    
    if 'occupancy' not in data.columns:
        data['occupancy'] = np.where((data['timestamp'].dt.hour >= 9) & 
                                    (data['timestamp'].dt.hour <= 17) & 
                                    (data['timestamp'].dt.dayofweek < 5),
                                    0.7 + 0.3 * np.random.rand(len(data)), 
                                    0.1 + 0.2 * np.random.rand(len(data)))
    
    # Select only needed columns
    energy_data = data[['timestamp', 'energy_consumption', 'outdoor_temperature', 'humidity', 'occupancy']]
    
    # Save the processed dataset
    energy_data.to_csv('building_energy_data.csv', index=False)
    print(f"Data saved to 'building_energy_data.csv' with {len(energy_data)} rows")
    return energy_data

def generate_synthetic_energy_data():
    """Generate synthetic building energy data"""
    print("Generating synthetic building energy data...")
    
    # Generate time series for 2 years of hourly data
    start_date = datetime(2021, 1, 1)
    end_date = datetime(2022, 12, 31, 23)
    dates = pd.date_range(start=start_date, end=end_date, freq='h')
    
    # Create DataFrame with timestamp
    df = pd.DataFrame({'timestamp': dates})
    
    # Basic patterns
    hour_of_day = df['timestamp'].dt.hour
    day_of_week = df['timestamp'].dt.dayofweek
    month_of_year = df['timestamp'].dt.month
    
    # Generate outdoor temperature with seasonal patterns
    seasonal_temp = 15 + 10 * np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi)
    daily_temp_variation = 5 * np.sin((hour_of_day / 24) * 2 * np.pi - np.pi/2)
    random_temp_variation = 3 * np.random.randn(len(df))
    df['outdoor_temperature'] = seasonal_temp + daily_temp_variation + random_temp_variation
    
    # Generate humidity
    df['humidity'] = 60 + 20 * np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi - np.pi/4) + 10 * np.random.randn(len(df))
    df['humidity'] = np.clip(df['humidity'], 20, 95)
    
    # Building occupancy (workdays 8am-6pm)
    is_workday = (day_of_week < 5)
    is_work_hours = (hour_of_day >= 8) & (hour_of_day <= 18)
    occupancy_base = np.where(is_workday & is_work_hours, 0.7, 0.1)
    df['occupancy'] = occupancy_base + 0.2 * np.random.random(len(df))
    
    # Energy consumption
    # Base load
    base_load = 20 + 5 * np.random.random(len(df))
    
    # HVAC load (more when temp is far from 21°C comfort temp)
    hvac_load = 5 + 0.5 * np.abs(df['outdoor_temperature'] - 21)**1.5 
    
    # Lighting & equipment (varies by hour and occupancy)
    lighting_equip = 5 + 20 * df['occupancy']
    
    # Seasonal adjustment (more energy in winter)
    winter_effect = 15 * (1 - np.sin((df['timestamp'].dt.dayofyear / 365) * 2 * np.pi))
    
    # Hour of day effect
    hour_effect = 10 * np.sin((hour_of_day / 24) * 2 * np.pi)
    
    # Weekend effect (less energy on weekends)
    weekend_effect = np.where(day_of_week >= 5, 0.7, 1.0)
    
    # Combine all effects
    df['energy_consumption'] = (base_load + hvac_load + lighting_equip + winter_effect + hour_effect) * weekend_effect
    
    # Add some random noise
    df['energy_consumption'] += 5 * np.random.randn(len(df))
    df['energy_consumption'] = np.maximum(df['energy_consumption'], 10)  # Ensure minimum value
    
    # Save to CSV
    df.to_csv('building_energy_data.csv', index=False)
    print(f"Synthetic data saved to 'building_energy_data.csv' with {len(df)} rows")
    return df

# Call the function to get the data
energy_data = get_building_energy_data()

# Display basic info
print(f"Dataset shape: {energy_data.shape}")
print(f"Date range: {energy_data['timestamp'].min()} to {energy_data['timestamp'].max()}")
print("\nData preview:")
print(energy_data.head())

Attempting to download data from Building Data Genome 2...
Failed to download from Building Data Genome 2 (Status code: 404)
Attempting to download data from ASHRAE Great Energy Predictor...
Failed to download from ASHRAE Great Energy Predictor (Status code: 404)
All data sources failed. Generating synthetic data instead.
Generating synthetic building energy data...
Synthetic data saved to 'building_energy_data.csv' with 17520 rows
Dataset shape: (17520, 5)
Date range: 2021-01-01 00:00:00 to 2022-12-31 23:00:00

Data preview:
            timestamp  outdoor_temperature   humidity  occupancy  \
0 2021-01-01 00:00:00            10.101548  31.553341   0.290471   
1 2021-01-01 01:00:00            12.444014  48.556798   0.189441   
2 2021-01-01 02:00:00            14.669119  67.169386   0.210637   
3 2021-01-01 03:00:00             5.419474  60.634946   0.140232   
4 2021-01-01 04:00:00            13.969947  50.327779   0.298341   

   energy_consumption  
0           75.770463  
1          